# Feature Flags and Progressive Rollout

**Phase 06** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-06/13-feature-flags-progressive-rollout.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic fastapi pydantic uvicorn

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

You have improved your prompt. In testing it looks better. You want to release it. The tempting path is to update the prompt and deploy. But what does "better" mean on real production traffic?

Your test set is not production. Your test users are not representative. Your evaluation covers the query patterns you thought to include, not the long tail of real queries your users actually send.

The cost of getting it wrong is high. Users start getting worse responses. They complain or churn. You roll back. Now you have a rollback incident that shakes confidence in the whole AI feature.

Feature flags let you release gradually. Instead of flipping all traffic to the new version at once, you expos...

## The Concept

### Traffic Routing at the Flag Layer



### Why Deterministic Hashing Matters

A naive approach uses `random.random() < 0.1` to route 10% of traffic. This means the same user might get variant A on one request and variant B on the next. That is not a controlled experiment: you cannot attribute behavior changes to the variant because the same user experiences both.

Deterministic hashing fixes this:

```
hash(user_id) % 100 < rollout_pct
```

The same `user_id` always produces the same hash, so the same user always gets the same variant for a given flag configuration. You can analyze by user c...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 13: Feature Flags and Progressive Rollout
Phase 06: Shipping

Three rollout modes for AI services:
  - Shadow: run new version in parallel, serve old to users, compare outputs
  - Canary: route X% of real traffic to new version by deterministic user hash
  - A/B: split traffic for outcome metric measurement

Usage:
    python main.py              # run the demo showing all three modes
    uvicorn main:app --reload   # start the FastAPI service

Requires: ANTHROPIC_API_KEY environment variable
"""

from __future__ import annotations

import hashlib
import logging
import time
from contextlib import asynccontextmanager
from dataclasses import dataclass
from enum import Enum
from typing import Optional

import anthropic
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

MODEL_ID = "claude-3-5-haiku-20241022"

# ---------------------------------------------------------------------------
# Prompt templates - in production these would be loaded from a file
# or the version manifest registry from Lesson 12.
# ---------------------------------------------------------------------------

PROMPTS: dict[str, str] = {
    "v1.0": "You are a helpful assistant. Answer concisely in 1-3 sentences.",
    "v1.1": (
        "You are a helpful assistant. Answer concisely in 1-3 sentences. "
        "End your response with a one-line summary starting with 'In short:'"
    ),
}

### Feature flag types

In [ ]:
class RolloutMode(str, Enum):
    SHADOW = "shadow"  # run new version in parallel, serve old to users
    CANARY = "canary"  # route X% of real traffic to new version
    AB = "ab"          # split traffic by user ID, measure outcome metric


@dataclass
class FeatureFlag:
    """
    Routes requests to prompt variants based on rollout_pct and mode.

    Attributes:
        name:        Unique flag identifier, included in the hash key so
                     different flags produce independent bucket assignments.
        rollout_pct: 0-100. Percentage of user IDs assigned to variant B.
        mode:        shadow, canary, or ab.
        variant_a:   The control prompt version (current production).
        variant_b:   The treatment prompt version (new version under test).
    """

    name: str
    rollout_pct: float
    mode: RolloutMode
    variant_a: str
    variant_b: str

    def _bucket(self, user_id: str) -> int:
        """
        Deterministic hash of user_id to a bucket 0-99.
        Including the flag name ensures different flags assign users independently.
        """
        key = f"{self.name}:{user_id}"
        digest = hashlib.md5(key.encode(), usedforsecurity=False).hexdigest()
        return int(digest[:8], 16) % 100

    def variant_for(self, user_id: str) -> str:
        """
        Return 'a' or 'b' for this user.
        Deterministic: same user_id always returns the same variant.
        """
        bucket = self._bucket(user_id)
        return "b" if bucket < self.rollout_pct else "a"

    def prompt_for(self, user_id: str) -> str:
        """Return the prompt version string for this user."""
        v = self.variant_for(user_id)
        return self.variant_b if v == "b" else self.variant_a

### Model call

In [ ]:
def call_model(prompt_version: str, user_message: str, model_id: str = MODEL_ID) -> dict:
    """
    Call Claude with the given prompt version.
    Returns response text, latency, and token counts.
    """
    client = anthropic.Anthropic()
    system = PROMPTS.get(prompt_version, PROMPTS["v1.0"])

    start = time.monotonic()
    response = client.messages.create(
        model=model_id,
        max_tokens=512,
        system=system,
        messages=[{"role": "user", "content": user_message}],
    )
    latency_ms = int((time.monotonic() - start) * 1000)

    return {
        "text": response.content[0].text,
        "prompt_version": prompt_version,
        "latency_ms": latency_ms,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }

### Shadow mode

In [ ]:
def run_shadow(
    flag: FeatureFlag,
    user_id: str,
    user_message: str,
    model_id: str = MODEL_ID,
) -> dict:
    """
    Shadow mode: call both variants.
    Return variant A response to the caller (this is what users see).
    Log both outputs for comparison. Variant B is never shown to users.
    """
    result_a = call_model(flag.variant_a, user_message, model_id)
    result_b = call_model(flag.variant_b, user_message, model_id)

    logger.info(
        "shadow_compare  flag=%s  user=%s  "
        "a_tokens=%d  b_tokens=%d  a_latency=%dms  b_latency=%dms",
        flag.name,
        user_id,
        result_a["output_tokens"],
        result_b["output_tokens"],
        result_a["latency_ms"],
        result_b["latency_ms"],
    )

    # Log truncated outputs at DEBUG level - these feed into your eval harness
    logger.debug("shadow A (%s): %s", flag.variant_a, result_a["text"][:150])
    logger.debug("shadow B (%s): %s", flag.variant_b, result_b["text"][:150])

    # Return A to the caller - B comparison data is for internal analysis only
    return {
        **result_a,
        "variant": "a",
        "shadow_b_text": result_b["text"],
        "shadow_b_tokens": result_b["output_tokens"],
        "shadow_b_latency_ms": result_b["latency_ms"],
    }

### Main router

In [ ]:
def route_request(
    flag: FeatureFlag,
    user_id: str,
    user_message: str,
    model_id: str = MODEL_ID,
) -> dict:
    """
    Route a request based on flag mode and user_id.

    shadow: run both variants, return A to caller, log B for comparison
    canary: deterministic routing, user sees the variant they are assigned
    ab:     same as canary, also logs variant assignment for outcome tracking
    """
    if flag.mode == RolloutMode.SHADOW:
        return run_shadow(flag, user_id, user_message, model_id)

    # Canary and A/B: user sees their assigned variant
    variant = flag.variant_for(user_id)
    prompt_version = flag.variant_b if variant == "b" else flag.variant_a
    result = call_model(prompt_version, user_message, model_id)
    result["variant"] = variant
    result["flag_name"] = flag.name

    if flag.mode == RolloutMode.AB:
        # Log assignment so it can be joined with downstream outcome metrics
        logger.info(
            "ab_assignment  flag=%s  user=%s  variant=%s  prompt=%s",
            flag.name,
            user_id,
            variant,
            prompt_version,
        )

    return result

### FastAPI service

In [ ]:
# Flag config - change mode and rollout_pct as the rollout progresses:
#   Shadow (week 1):  mode=SHADOW, rollout_pct=0
#   10% canary:       mode=CANARY, rollout_pct=10
#   50% canary:       mode=CANARY, rollout_pct=50
#   Full rollout:     remove flag, use variant_b directly
ACTIVE_FLAG = FeatureFlag(
    name="prompt-v1.1-rollout",
    rollout_pct=10.0,
    mode=RolloutMode.SHADOW,
    variant_a="v1.0",
    variant_b="v1.1",
)


@asynccontextmanager
async def lifespan(app: FastAPI):
    """Log the active flag configuration at startup."""
    flag = ACTIVE_FLAG
    logger.info("=== FEATURE FLAG ACTIVE ===")
    logger.info("name:        %s", flag.name)
    logger.info("mode:        %s", flag.mode)
    logger.info("rollout_pct: %.0f%%", flag.rollout_pct)
    logger.info("variant_a:   %s", flag.variant_a)
    logger.info("variant_b:   %s", flag.variant_b)
    logger.info("===========================")
    app.state.flag = flag
    yield


app = FastAPI(title="AI Service with Feature Flags", lifespan=lifespan)


class ChatRequest(BaseModel):
    user_id: str
    message: str


class ChatResponse(BaseModel):
    response: str
    variant: str
    prompt_version: str
    latency_ms: int


@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    """
    Chat endpoint with flag-based routing.

    In shadow mode: all users see variant A response.
    In canary/ab mode: users see their deterministically assigned variant.
    """
    flag = app.state.flag

    try:
        result = route_request(
            flag=flag,
            user_id=request.user_id,
            user_message=request.message,
            model_id=MODEL_ID,
        )
    except anthropic.APIError as exc:
        raise HTTPException(status_code=502, detail=str(exc)) from exc

    return ChatResponse(
        response=result["text"],
        variant=result.get("variant", "a"),
        prompt_version=result["prompt_version"],
        latency_ms=result["latency_ms"],
    )


@app.get("/flag-status")
async def flag_status():
    """Returns the active flag configuration. Useful for debugging routing."""
    flag = app.state.flag
    return {
        "name": flag.name,
        "mode": flag.mode,
        "rollout_pct": flag.rollout_pct,
        "variant_a": flag.variant_a,
        "variant_b": flag.variant_b,
    }


@app.get("/flag-preview/{user_id}")
async def flag_preview(user_id: str):
    """Show which variant a given user_id maps to without making an API call."""
    flag = app.state.flag
    return {
        "user_id": user_id,
        "variant": flag.variant_for(user_id),
        "prompt_version": flag.prompt_for(user_id),
        "mode": flag.mode,
    }

### Demo: run from command line to see all three modes

In [ ]:
def demo_distribution(flag: FeatureFlag, n: int = 1000) -> None:
    """Check that bucket distribution is approximately uniform."""
    counts = {"a": 0, "b": 0}
    for i in range(n):
        v = flag.variant_for(f"user-{i:05d}")
        counts[v] += 1
    pct_b = counts["b"] / n * 100
    print(f"  Distribution over {n} users: A={counts['a']} ({100-pct_b:.1f}%) B={counts['b']} ({pct_b:.1f}%)")
    print(f"  Expected: B={flag.rollout_pct:.0f}%  Actual: B={pct_b:.1f}%")

### Demo

In [ ]:
print("=== FeatureFlag Demo ===\n")

# Show deterministic assignment
print("1. Deterministic variant assignment (same user always gets same variant):")
flag = FeatureFlag(
    name="demo-flag",
    rollout_pct=20.0,
    mode=RolloutMode.CANARY,
    variant_a="v1.0",
    variant_b="v1.1",
)
test_users = ["user-001", "user-042", "user-007", "user-999", "user-001"]
for uid in test_users:
    variant = flag.variant_for(uid)
    prompt = flag.prompt_for(uid)
    print(f"  {uid} -> variant={variant} prompt={prompt}")

print()
print("2. Bucket distribution at 20% rollout:")
demo_distribution(flag, n=10000)

print()
print("3. Routing modes:")
for mode in [RolloutMode.SHADOW, RolloutMode.CANARY, RolloutMode.AB]:
    test_flag = FeatureFlag(
        name=f"mode-test-{mode.value}",
        rollout_pct=50.0,
        mode=mode,
        variant_a="v1.0",
        variant_b="v1.1",
    )
    v = test_flag.variant_for("user-100")
    print(f"  mode={mode.value:<8}  user-100 -> variant={v}  (B response shown to user: {mode != RolloutMode.SHADOW and v == 'b'})")

print()
print("4. Shadow mode note:")
print("  In shadow mode, both prompts run but only variant_a is returned to users.")
print("  Set logging level to DEBUG to see shadow comparison output.")
print("  Use those logs to evaluate whether variant_b is ready for canary.")

---

## Self-Check

Answer these without running code first:

**1. What is the recommended order and why?**

- A. Deploy directly - your eval suite already validated it
- B. Go straight to 10% canary - shadow mode wastes API calls on outputs users never see
- C. Run shadow mode first to validate on real traffic with zero user risk, then promote to canary
- D. Run an A/B test before shadow mode to measure business metrics first

**2. What is the root cause and how do you fix it?**

- A. The random seed was not set; fix with random.seed(42)
- B. The 10% threshold is too low; increase to 20% for stable assignment
- C. random.random() produces a different value on each call, so users can land in different variants across requests; fix with deterministic hashing: hash(user_id) % 100 < 10
- D. Users should be assigned variants in the database, not at request time

**3. Is the colleague correct, and what would indicate shadow mode is broken?**

- A. Incorrect - in shadow mode users should see variant B 50% of the time
- B. Correct - in shadow mode all users see variant A responses; variant B runs in the background for comparison only
- C. Incorrect - the prompt_version should be hidden from the HTTP response entirely
- D. Correct, but only if the b_tokens count in the logs is also zero

**4. What determines the independence of bucket assignments across flags?**

- A. Yes, the same user always falls in the treatment group for all flags at the same rollout percentage
- B. No, because each flag includes its name in the hash key: hash(flag_name + ':' + user_id) produces independent buckets per flag
- C. Yes, because MD5 hashing always produces consistent results for the same user
- D. No, because different flags use different random number generators

**5. What metrics should you look at from the canary cohort before expanding to 25%?**

- A. Only token usage - lower token counts mean the new prompt is more efficient
- B. User error rate, support tickets or escalations from the treatment cohort, and your eval suite run against a sample of variant B responses
- C. The model provider latency graphs for variant B requests
- D. Whether variant B users are sending more requests than variant A users

**6. What is the correct practice for handling flags after full rollout?**

- A. Set rollout_pct=100 and leave the flag permanently - this is the safest configuration
- B. Remove the flag from the codebase and make variant B the default directly - dead flags create confusion and maintenance debt
- C. Archive the flag by setting its mode to SHADOW permanently
- D. Document the flag in a README so future engineers know its state

**7. **

- A. Shadow mode - run both versions and compare thumbs up rates in the background
- B. A/B mode - log variant assignment per user_id so you can join with the thumbs up event data to compute rate by cohort
- C. Canary mode - route 50% to variant B and count thumbs up overall
- D. You do not need a feature flag for this; just deploy variant B and compare before/after averages

**8. How do you evaluate whether 8.47% is within acceptable range for a 10% flag?**

- A. It is not acceptable - the hash function must be replaced because the error exceeds 1 percentage point
- B. It is acceptable - with 10,000 samples and a 10% target, statistical variation of up to 1-2 percentage points is expected; run 100,000 samples to confirm it converges near 10%
- C. It is not acceptable - MD5 is a security hash and not suitable for bucketing
- D. It is acceptable as long as the flag is in shadow mode

_Answers are in `checks.json` in the lesson directory._